# Using SGD with NAG

In [1]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("/kaggle/input/datasets/ashura369/my-dataset-2/heart.csv")

In [3]:
df.head(6)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
5,39,M,NAP,120,339,0,Normal,170,N,0.0,Up,0


In [4]:
df.select_dtypes(include='object').nunique()

Sex               2
ChestPainType     4
RestingECG        3
ExerciseAngina    2
ST_Slope          3
dtype: int64

In [5]:
df['Sex'].unique()

array(['M', 'F'], dtype=object)

In [6]:
# lowercasing all the columns names

df.columns = df.columns.str.lower()

In [7]:
# checking for null values
df.isna().sum().sum()

np.int64(0)

In [8]:
df.select_dtypes(include=['float', 'int']).describe()

,age,restingbp,cholesterol,fastingbs,maxhr,oldpeak,heartdisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


In [9]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer

In [10]:
transform = ColumnTransformer(
    transformers=[
        ('t1', MinMaxScaler(), ['fastingbs']),
        ('t2', StandardScaler(), ['age', 'restingbp', 'cholesterol', 'maxhr', 'oldpeak'])
    ], remainder='drop'
)

temp = transform.fit_transform(df)

In [11]:
cols = transform.get_feature_names_out().tolist()
cols = [i.split("__")[-1] for i in cols]
cols

['fastingbs', 'age', 'restingbp', 'cholesterol', 'maxhr', 'oldpeak']

In [12]:
temp = pd.DataFrame(temp, columns=cols)
temp.head()

,fastingbs,age,restingbp,cholesterol,maxhr,oldpeak
0,0.0,-1.433140,0.410909,0.825070,1.382928,-0.832432
1,0.0,-0.478484,1.491752,-0.171961,0.754157,0.105664
2,0.0,-1.751359,-0.129513,0.770188,-1.525138,-0.832432
3,0.0,-0.584556,0.302825,0.139040,-1.132156,0.574711
4,0.0,0.051881,0.951331,-0.034755,-0.581981,-0.832432


In [13]:
df.update(temp)
df.head(6)

,age,sex,chestpaintype,restingbp,cholesterol,fastingbs,restingecg,maxhr,exerciseangina,oldpeak,st_slope,heartdisease
0,-1.433140,M,ATA,0.410909,0.825070,0,Normal,1.382928,N,-0.832432,Up,0
1,-0.478484,F,NAP,1.491752,-0.171961,0,Normal,0.754157,N,0.105664,Flat,1
2,-1.751359,M,ATA,-0.129513,0.770188,0,ST,-1.525138,N,-0.832432,Up,0
3,-0.584556,F,ASY,0.302825,0.139040,0,Normal,-1.132156,Y,0.574711,Flat,1
4,0.051881,M,NAP,0.951331,-0.034755,0,Normal,-0.581981,N,-0.832432,Up,0
5,-1.539213,M,NAP,-0.669935,1.282424,0,Normal,1.304332,N,-0.832432,Up,0


In [14]:
# using one hot encoding

temp = pd.get_dummies(df.select_dtypes(include='object'), dtype=int)

temp.sample(3)

,sex_F,sex_M,chestpaintype_ASY,chestpaintype_ATA,chestpaintype_NAP,chestpaintype_TA,restingecg_LVH,restingecg_Normal,restingecg_ST,exerciseangina_N,exerciseangina_Y,st_slope_Down,st_slope_Flat,st_slope_Up
606,0,1,1,0,0,0,0,0,1,0,1,0,1,0
229,1,0,1,0,0,0,0,0,1,1,0,0,0,1
473,0,1,0,0,1,0,0,0,1,0,1,0,1,0


In [15]:
data = df.drop(columns=(df.select_dtypes(include='object')))
data = pd.concat([data, temp], axis=1)
data.sample(3)

,age,restingbp,cholesterol,fastingbs,maxhr,oldpeak,heartdisease,sex_F,sex_M,chestpaintype_ASY,...,chestpaintype_NAP,chestpaintype_TA,restingecg_LVH,restingecg_Normal,restingecg_ST,exerciseangina_N,exerciseangina_Y,st_slope_Down,st_slope_Flat,st_slope_Up
460,0.370100,0.356867,0.715305,1,-0.739174,0.949950,1,0,1,1,...,0,0,0,0,1,0,1,0,1,0
504,0.900464,1.383668,0.102451,1,-0.974963,1.981855,1,0,1,1,...,0,0,0,1,0,0,1,1,0,0
291,-0.690629,0.410909,0.532364,0,-0.071105,0.105664,0,1,0,0,...,0,0,0,1,0,1,0,0,0,1


In [16]:
from sklearn.model_selection import train_test_split as ttt

In [17]:
x = data.drop(columns=['heartdisease'])
y = data[['heartdisease']]

x_train, x_test, y_train, y_test = ttt(x, y, test_size=0.2, random_state=42)

In [18]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, losses, callbacks, regularizers

2026-05-30 16:09:46.553147: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780157386.756833      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780157386.819145      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780157387.301867      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780157387.301909      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780157387.301912      58 computation_placer.cc:177] computation placer alr

In [19]:
tf.random.set_seed(42)

model = keras.Sequential([
    layers.Dense(1, activation='sigmoid', input_shape=(x_train.shape[1],))
])

opt = optimizers.SGD(learning_rate=0.005, momentum=0.9)
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping callback (equivalent to early_stopping=True, n_iter_no_change=15)
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# Train the model (using validation_split for validation monitoring)
model.fit(
    x_train, y_train, 
    epochs=1000, 
    validation_split=0.1, 
    callbacks=[early_stop], 
    verbose=0
)

# Convert Keras output probabilities to binary class labels (1D array)
prediction = (model.predict(x_test) > 0.5).astype(int).flatten()

I0000 00:00:1780157399.576852      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13615 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780157399.582038      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1780157401.709477     138 service.cc:152] XLA service 0x7a3d340275f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780157401.709512     138 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1780157401.709516     138 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1780157401.893943     138 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1780157402.194520     138 device_compiler.h:188] Compiled clust

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step


In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

print(f"ACCURACY SCORE : \n{accuracy_score(y_test, prediction)}\n")
print(f"PRECISION SCORE : \n{precision_score(y_test, prediction, average='weighted')}\n")
print(f"RECALL SCORE : \n{recall_score(y_test, prediction, average='weighted')}\n")
print(f"CONFUSION MATRIX : \n{confusion_matrix(y_test, prediction)}")

ACCURACY SCORE : 
0.8641304347826086

PRECISION SCORE : 
0.866437163412555

RECALL SCORE : 
0.8641304347826086

CONFUSION MATRIX : 
[[67 10]
 [15 92]]


In [21]:
tf.random.set_seed(42)

model2 = keras.Sequential([
    layers.Dense(1, activation='sigmoid', kernel_regularizer=regularizers.l1(0.01), input_shape=(x_train.shape[1],))
])

# SGDClassifier doesn't use momentum by default, just standard SGD
opt2 = optimizers.SGD(learning_rate=0.01) 
model2.compile(optimizer=opt2, loss='binary_crossentropy', metrics=['accuracy'])

early_stop2 = callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

model2.fit(
    x_train, y_train, 
    epochs=1000, 
    validation_split=0.1, 
    callbacks=[early_stop2], 
    verbose=0
)

prediction2 = (model2.predict(x_test) > 0.5).astype(int).flatten()

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step


In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

print(f"ACCURACY SCORE : \n{accuracy_score(y_test, prediction2)}\n")
print(f"PRECISION SCORE : \n{precision_score(y_test, prediction2, average='weighted')}\n")
print(f"RECALL SCORE : \n{recall_score(y_test, prediction2, average='weighted')}\n")
print(f"CONFUSION MATRIX : \n{confusion_matrix(y_test, prediction2)}")

ACCURACY SCORE : 
0.8369565217391305

PRECISION SCORE : 
0.8438474061938103

RECALL SCORE : 
0.8369565217391305

CONFUSION MATRIX : 
[[67 10]
 [20 87]]


## Using NAG

In [23]:
tf.random.set_seed(42)

model3 = keras.Sequential([
    layers.Dense(1, activation='sigmoid', input_shape=(x_train.shape[1],))
])

# Setting nesterov=True for NAG (Nesterov Accelerated Gradient)
opt3 = optimizers.SGD(learning_rate=0.005, momentum=0.9, nesterov=True)
model3.compile(optimizer=opt3, loss='binary_crossentropy', metrics=['accuracy'])

early_stop3 = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

model3.fit(
    x_train, y_train, 
    epochs=1000, 
    validation_split=0.1, 
    callbacks=[early_stop3], 
    verbose=0
)

prediction3 = (model3.predict(x_test) > 0.5).astype(int).flatten()

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step


In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

print(f"ACCURACY SCORE : \n{accuracy_score(y_test, prediction3)}\n")
print(f"PRECISION SCORE : \n{precision_score(y_test, prediction3, average='weighted')}\n")
print(f"RECALL SCORE : \n{recall_score(y_test, prediction3, average='weighted')}\n")
print(f"CONFUSION MATRIX : \n{confusion_matrix(y_test, prediction3)}")

ACCURACY SCORE : 
0.8260869565217391

PRECISION SCORE : 
0.8310470433232581

RECALL SCORE : 
0.8260869565217391

CONFUSION MATRIX : 
[[65 12]
 [20 87]]
